# Run OpenFold Structure Module — TEST SET ONLY

Runs the structure module for **all test-set proteins** (from `training_info.json`). Writes one PDB per test protein to `sae_pdbs/`.

In [ ]:
# Cell 1: Add OpenFold to path (run this first) — set to the directory that contains the 'openfold' package
import sys
import os

OPENFOLD_PATH = os.environ.get("OPENFOLD_PATH", "/storage/scratch1/5/sshrestha304/Autoencoder/CompleteProteins/openfold")
if OPENFOLD_PATH != "/path/to/openfold" and os.path.isdir(OPENFOLD_PATH):
    parent = os.path.dirname(OPENFOLD_PATH) if os.path.basename(OPENFOLD_PATH) == "openfold" else OPENFOLD_PATH
    if parent not in sys.path:
        sys.path.insert(0, parent)
    print(f"Added to path: {parent}")
else:
    print("Set OPENFOLD_PATH to the folder containing the 'openfold' package (e.g. where you cloned OpenFold).")
    print("Or: pip install openfold")
    print("Or in terminal: export OPENFOLD_PATH=/path/to/openfold")

In [ ]:
# Cell 2: Imports
import json
import numpy as np
import torch

from openfold.config import model_config
from openfold.model.model import AlphaFold
from openfold.np import protein, residue_constants
from openfold.utils.import_weights import import_openfold_weights_
from openfold.utils.feats import atom14_to_atom37

print("✅ Imports OK")

In [ ]:
# Cell 2: Configuration — TEST SET ONLY (from training_info.json)
BASE = "/storage/scratch1/5/sshrestha304/Autoencoder/CompleteProteins"
LAYER = 47
training_info_path = "sae_results/training_info.json"
single_from_pair = True
openfold_checkpoint_path = None
# Structure module: use CPU to avoid 'no kernel image' (OpenFold built for sm_80 only; fails on Blackwell/Ada)
model_device = "cpu"
config_name = "model_1_ptm"

with open(training_info_path) as f:
    training_config = json.load(f)
TEST_PROTEIN_IDS = set(training_config.get("test_protein_ids") or [])
if not TEST_PROTEIN_IDS and training_config.get("test_proteins"):
    TEST_PROTEIN_IDS = {p.replace(f"_pair_block_{LAYER}.npy", "").replace(".npy", "") for p in training_config["test_proteins"]}
if not TEST_PROTEIN_IDS:
    raise ValueError("No test_protein_ids in training_info.json")

output_dir = os.path.join(BASE, "sae_pdbs")
os.makedirs(output_dir, exist_ok=True)
print(f"Test proteins: {len(TEST_PROTEIN_IDS)}")
print(f"Output dir: {output_dir}")

In [ ]:
# Cell 3: Helpers
def sequence_to_aatype(seq):
    return np.array([residue_constants.restype_order.get(c, residue_constants.restype_num) for c in seq.upper()], dtype=np.int64)

def load_sequence_from_fasta(path):
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith(">")]
    return "".join(lines)

In [ ]:
# Cell 4: Load OpenFold model once
config = model_config(config_name)
model = AlphaFold(config)
if openfold_checkpoint_path and os.path.exists(openfold_checkpoint_path):
    ckpt = openfold_checkpoint_path
else:
    ckpt = os.path.join(OPENFOLD_PATH, "resources", "openfold_params", "finetuning_ptm_1.pt")
if not os.path.exists(ckpt):
    raise FileNotFoundError(
        f"Checkpoint not found: {ckpt}\n"
        "Download (on cluster): mkdir -p " + os.path.join(OPENFOLD_PATH, "resources", "openfold_params") + "\n"
        "  wget -O " + ckpt + " 'https://huggingface.co/nz/OpenFold/resolve/main/finetuning_ptm_1.pt'\n"
        "See DOWNLOAD_OPENFOLD_CHECKPOINT.md for other options."
    )
d = torch.load(ckpt, map_location="cpu")
if "ema" in d: d = d["ema"]["params"]
elif "state_dict" in d: d = d["state_dict"]
elif "module" in d: d = {k.replace("module.", ""): v for k, v in d["module"].items()}
import_openfold_weights_(model, d)
model = model.to(model_device).eval()
inplace_safe = model_device != "cpu"
c_s = 384
print("✅ Model loaded")

In [ ]:
# Cell 5: Run structure module for each TEST protein
for i, protein_id in enumerate(sorted(TEST_PROTEIN_IDS), 1):
    pair_npy = os.path.join(BASE, "sae_reconstructions", f"{protein_id}_reconstructed_pair_block_{LAYER}.npy")
    if not os.path.isfile(pair_npy):
        print(f"[{i}/{len(TEST_PROTEIN_IDS)}] ⏭️ Skip {protein_id} (no pair file)")
        continue
    subdir = os.path.join(BASE, protein_id)
    single_npy = os.path.join(subdir, f"single_block_{LAYER}.npy")
    aatype_npy = os.path.join(subdir, f"aatype_{LAYER}.npy")
    fasta_path = None
    for fn in [f"{protein_id}.fasta", "seq.fasta"]:
        fp = os.path.join(subdir, fn)
        if os.path.isfile(fp): fasta_path = fp; break
    if not fasta_path and os.path.isdir(subdir):
        for f in os.listdir(subdir):
            if f.endswith(".fasta"): fasta_path = os.path.join(subdir, f); break

    pair = np.load(pair_npy, allow_pickle=True)
    if pair.ndim == 4: pair = pair[0]
    n_res, c_z = pair.shape[0], pair.shape[2]
    if c_z != 128:
        pair = np.concatenate([pair, np.zeros((n_res, n_res, 128 - c_z), dtype=pair.dtype)], axis=2) if c_z < 128 else pair[:, :, :128].copy()

    if single_npy and os.path.isfile(single_npy):
        single = np.load(single_npy, allow_pickle=True)
        if single.ndim == 3: single = single[0]
        if single.shape != (n_res, c_s): single = np.concatenate([single, np.zeros((n_res, c_s - single.shape[1]), dtype=single.dtype)], axis=1) if single.shape[1] < c_s else single[:, :c_s]
    else:
        single = pair.mean(axis=1)
        single = np.concatenate([single, np.zeros((n_res, c_s - single.shape[1]), dtype=single.dtype)], axis=1) if single.shape[1] < c_s else single[:, :c_s].copy()

    if fasta_path and os.path.isfile(fasta_path):
        aatype = sequence_to_aatype(load_sequence_from_fasta(fasta_path))
    elif aatype_npy and os.path.isfile(aatype_npy):
        aatype = np.load(aatype_npy, allow_pickle=True).astype(np.int64)
    else:
        # Dummy aatype (all alanine) when only .npy available — TM-score unaffected (uses backbone only)
        aatype = np.zeros(n_res, dtype=np.int64)
    if len(aatype) != n_res:
        print(f"[{i}] ⏭️ Skip {protein_id} (aatype len != n_res)")
        continue

    pair_b = np.expand_dims(pair, 0)
    single_b = np.expand_dims(single, 0)
    aatype_b = np.expand_dims(aatype, 0)
    mask_t = torch.ones((1, n_res), dtype=torch.float32, device=model_device)
    evoformer_output = {"single": torch.from_numpy(single_b).float().to(model_device), "pair": torch.from_numpy(pair_b).float().to(model_device)}
    aatype_t = torch.from_numpy(aatype_b).long().to(model_device)

    with torch.no_grad():
        sm_out = model.structure_module(evoformer_output, aatype_t, mask=mask_t, inplace_safe=inplace_safe)
    atom14_pos = sm_out["positions"][-1]
    feats = {"residx_atom37_to_atom14": torch.from_numpy(np.take(residue_constants.RESTYPE_ATOM37_TO_ATOM14, aatype, axis=0)).long().to(model_device).unsqueeze(0), "atom37_atom_exists": torch.from_numpy(np.take(residue_constants.RESTYPE_ATOM37_MASK, aatype, axis=0)).float().to(model_device).unsqueeze(0)}
    atom37_pos = atom14_to_atom37(atom14_pos, feats)
    result = {"final_atom_positions": atom37_pos[0].cpu().numpy(), "final_atom_mask": feats["atom37_atom_exists"][0].cpu().numpy()}
    unprocessed_features = {"aatype": np.expand_dims(aatype, 0), "residue_index": np.expand_dims(np.arange(n_res), 0)}
    pdb_str = protein.to_pdb(protein.from_prediction(unprocessed_features, result))
    out_path = os.path.join(output_dir, f"{protein_id}_predicted.pdb")
    with open(out_path, "w") as f:
        f.write(pdb_str)
    print(f"[{i}/{len(TEST_PROTEIN_IDS)}] ✅ {protein_id} -> {out_path}")

print("✅ Done. PDBs in", output_dir)